The script is based on DeepLearning.ai course materials and adapted to using Gemini AI instead of OpenAI.

!pip install crewai google-genai

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Agent, Task, Crew, LLM

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
gemini_llm = LLM(
    model = "gemini/gemini-2.5-flash",
    api_key = os.getenv("GEMINI_API_KEY")
)

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a newspaper column"
              "about the topic: {topic}."
              "You collect information from "
              "traditional and social media "
              "to help the audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic "
              "and use a broad range of sources "
              "targeting different demographic audiences. "
              "You should also provide examples "
              "and quotations from thos enegaged in "
              "in the topic of hate following.",
    allow_delegation=False,
	verbose=True,
    llm = gemini_llm
)

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a newspaper column about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    llm = gemini_llm
)

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are the chef editor to write a column "
              "on a {topic} provided by the Content Writer. "
              "Your goal is to ensure that the post follows "
              "the journalistic best practices and "
              "provides balanced viewpoints "
              "while highlighting the dangers "
              "and harms around the {topic}. ",
    allow_delegation=False,
    verbose=True,
    llm = gemini_llm
)

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest subjects, key players, "
            "and latest trends on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
    llm = gemini_llm
)

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "column post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
    llm = gemini_llm
)

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written but provocative column post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor,
    llm = gemini_llm
)

In [ ]:
crew = Crew(
    agents = [planner, writer, editor],
    tasks = [plan, write, edit],
    verbose = True,
    memory = False
)


In [ ]:
result = await crew.kickoff_async(
    inputs={"topic": "Hate Following on Social Media"}
)